In [81]:
import pandas as pd
import glob
import os
import urllib.request

folder_path = "../data/Premier_League/Not_merged"
url_live = "https://www.football-data.co.uk/mmz4281/2526/E0.csv"
live_file_path = os.path.join(folder_path, "E0_25_26_LIVE.csv")
urllib.request.urlretrieve(url_live, live_file_path)

file_pattern = os.path.join(folder_path, "*.csv")
files = glob.glob(file_pattern)

print(f"Found {len(files)} files")

Found 26 files


*TODO: POBIERANIE NIE TYLKO 2026 ALE WSZYSTKICH DANYCH Z LAT OD JAKICHS 2000 DO NAJNOWSZYCH*


In [82]:
core_columns =[
    'Div', 'Date', 'HomeTeam', 'AwayTeam',
    'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR',
    'Referee',
    'HS', 'AS', 'HST', 'AST', 'HC', 'AC',
    'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]

all_matches = []

for file in files:
    df = pd.read_csv(file, encoding='utf-8', on_bad_lines='skip')

    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce', format='mixed')

    cols_to_keep = [col for col in core_columns if col in df.columns]
    df = df[cols_to_keep]
    all_matches.append(df)


In [83]:
master_db = pd.concat(all_matches, ignore_index=True)

In [84]:
master_db = master_db.dropna(subset=['HomeTeam', 'AwayTeam'])
master_db=master_db.sort_values(by='Date').reset_index(drop=True)

In [85]:
def get_season(date):
    if pd.isna(date):
        return "Unknown"
    year = date.year
    month = date.month
    if month >= 7:
        return f"{year}/{year+1}"
    else:
        return f"{year-1}/{year}"

master_db['Season'] = master_db['Date'].apply(get_season)

In [86]:
output_dir = "../data/Premier_League"
output_filename = os.path.join(output_dir, "PremierLeague_WszystkieSezony.csv")

master_db.to_csv(output_filename, index=False)

**DANE Z PREMIER LEAGUE ZOSTALY SCALONE, WSTEPNIE OBROBIONE I WRZUCONE DO JEDNEGO PLIKU**

In [87]:
df=master_db.copy()
df.isna().sum()

Div         0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
Referee     0
HS          0
AS          0
HST         0
AST         0
HC          0
AC          0
HF          0
AF          0
HY          0
AY          0
HR          0
AR          0
Season      0
dtype: int64

In [88]:
df = master_db.copy()
ftr_mapping = {'H': 2, 'D': 1, 'A': 0}
df['Target'] = df['FTR'].map(ftr_mapping)

In [89]:
def get_home_points(ftr):
    if ftr == 'H': return 3
    elif ftr == 'D': return 1
    else: return 0

def get_away_points(ftr):
    if ftr == 'A': return 3
    elif ftr == 'D': return 1
    else: return 0

In [90]:
df['HomePoints'] = df['FTR'].apply(get_home_points)
df['AwayPoints'] = df['FTR'].apply(get_away_points)

print(df[['Date', 'HomeTeam', 'AwayTeam','FTR', 'HomePoints', 'AwayPoints']].head(10))

        Date    HomeTeam       AwayTeam FTR  HomePoints  AwayPoints
0 2000-08-19       Leeds        Everton   H           3           0
1 2000-08-19   Liverpool       Bradford   H           3           0
2 2000-08-19  Sunderland        Arsenal   H           3           0
3 2000-08-19    Charlton       Man City   H           3           0
4 2000-08-19     Chelsea       West Ham   H           3           0
5 2000-08-19    Coventry  Middlesbrough   A           0           3
6 2000-08-19       Derby    Southampton   D           1           1
7 2000-08-19   Tottenham        Ipswich   H           3           0
8 2000-08-19   Leicester    Aston Villa   D           1           1
9 2000-08-20  Man United      Newcastle   H           3           0


In [91]:
df['Date']= pd.to_datetime(df['Date'])
df = df.sort_values(by='Date').reset_index(drop=True)

df['HomeTeam'] = df['HomeTeam'].astype(str).str.split('_').str[0]
df['AwayTeam'] = df['AwayTeam'].astype(str).str.split('_').str[0]

home_dates = df[['Date', 'HomeTeam']].rename(columns={'HomeTeam':'Team'})
away_dates = df[['Date', 'AwayTeam']].rename(columns={'AwayTeam':'Team'})
all_matches = pd.concat([home_dates, away_dates]).sort_values(by=['Team','Date']).reset_index(drop=True)

all_matches['Raw_DaysOfRest']= all_matches.groupby('Team')['Date'].diff().dt.days
all_matches['Is_New_Spell'] = (all_matches['Raw_DaysOfRest'] > 300).astype(int)
all_matches['Spell_ID']= all_matches.groupby('Team')['Is_New_Spell'].cumsum()

all_matches['Team_Spell'] = all_matches['Team'] + '_' + all_matches['Spell_ID'].astype(str)

all_matches['DaysOfRest'] = all_matches['Raw_DaysOfRest'].apply(lambda x: 14.0 if pd.isna(x) or x>70 else x)



In [92]:
df = df.merge(all_matches[['Date','Team','DaysOfRest', 'Team_Spell']], left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left')
df.rename(columns={'DaysOfRest':'Home_DaysOfRest'}, inplace=True)

df['HomeTeam']=df['Team_Spell']
df.drop(['Team','Team_Spell'], axis=1, inplace=True)

df = df.merge(all_matches[['Date','Team', 'DaysOfRest', 'Team_Spell']], left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left')
df.rename(columns={'DaysOfRest':'Away_DaysOfRest'}, inplace=True)
df['AwayTeam']=df['Team_Spell']
df.drop(['Team','Team_Spell'], axis=1, inplace=True)

In [93]:
df=df.sort_values(by='Date').reset_index(drop=True)

#home stats
df['Home_Last5_HomePts']=df.groupby('HomeTeam')['HomePoints'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

#offensive home stats
df['Home_Last5_GF']=df.groupby('HomeTeam')['FTHG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShots']=df.groupby('HomeTeam')['HS'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShotsOT']=df.groupby('HomeTeam')['HST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_Corners']=df.groupby('HomeTeam')['HC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeFouls']=df.groupby('HomeTeam')['HF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeYellows']=df.groupby('HomeTeam')['HY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeReds']=df.groupby('HomeTeam')['HR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

#defensive home stats
df['Home_Last5_GA']=df.groupby('HomeTeam')['FTAG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShotsAgainst']=df.groupby('HomeTeam')['AS'].transform(lambda x:x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShotsOTAgainst']=df.groupby('HomeTeam')['AST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_CornersAgainst']=df.groupby('HomeTeam')['AC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeFoulsAgainst']=df.groupby('HomeTeam')['AF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeYellowsAgainst']=df.groupby('HomeTeam')['AY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeRedsAgainst']=df.groupby('HomeTeam')['AR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

In [94]:
#away stats
df['Away_Last5_AwayPts']=df.groupby('AwayTeam')['AwayPoints'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

#away offensive stats
df['Away_Last5_GF']=df.groupby('AwayTeam')['FTAG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShots']=df.groupby('AwayTeam')['AS'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShotsOT']=df.groupby('AwayTeam')['AST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_Corners']=df.groupby('AwayTeam')['AC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayFouls']=df.groupby('AwayTeam')['AF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayYellows']=df.groupby('AwayTeam')['AY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayReds']=df.groupby('AwayTeam')['AR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
#away defensive stats
df['Away_Last5_GA']=df.groupby('AwayTeam')['FTHG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShotsAgainst']=df.groupby('AwayTeam')['HS'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShotsOTAgainst']=df.groupby('AwayTeam')['HST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_CornersAgainst']=df.groupby('AwayTeam')['HC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayFoulsAgainst']=df.groupby('AwayTeam')['HF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayYellowsAgainst']=df.groupby('AwayTeam')['HY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayRedsAgainst']=df.groupby('AwayTeam')['HR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

In [95]:
home_form=df[['Date','HomeTeam','HomePoints','FTHG','FTAG','HS','AS','HST','AST','HC','AC','HF','AF','HY','AY','HR','AR']].copy()
home_form.columns = ['Date', 'Team', 'Points', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'ShotsOTFor', 'ShotsOTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'YellowsFor', 'YellowsAgainst', 'RedsFor', 'RedsAgainst']

away_form = df[['Date', 'AwayTeam', 'AwayPoints', 'FTAG', 'FTHG', 'AS', 'HS', 'AST', 'HST', 'AC', 'HC', 'AF', 'HF', 'AY', 'HY', 'AR', 'HR']].copy()
away_form.columns = ['Date', 'Team', 'Points', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'ShotsOTFor', 'ShotsOTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'YellowsFor', 'YellowsAgainst', 'RedsFor', 'RedsAgainst']


In [96]:
team_form=pd.concat([home_form, away_form]).sort_values(by=['Team','Date']).reset_index(drop=True)
stat_columns = ['Points', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'ShotsOTFor', 'ShotsOTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'YellowsFor', 'YellowsAgainst', 'RedsFor', 'RedsAgainst']
for col in stat_columns:
    team_form[f'Overall_Last5_{col}']=team_form.groupby('Team')[col].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

cols_to_keep = ['Date', 'Team'] + [f'Overall_Last5_{col}' for col in stat_columns]

team_form=team_form[cols_to_keep]

In [97]:
df = df.merge(team_form, left_on=['Date','HomeTeam'], right_on=['Date','Team'], how='left')
df.rename(columns=lambda x: f"Home_{x}" if x in team_form.columns and x not in ['Date', 'Team'] else x, inplace=True)
df.drop('Team',axis=1,inplace=True)

df = df.merge(team_form, left_on=['Date','AwayTeam'], right_on=['Date','Team'], how='left')
df.rename(columns=lambda x: f"Away_{x}" if x in team_form.columns and x not in ['Date', 'Team'] else x, inplace=True)
df.drop('Team',axis=1,inplace=True)


In [98]:
kolumny_do_podgladu = [
    'Date', 'HomeTeam', 'AwayTeam',
    'Home_Last5_HomePts', 'Away_Last5_AwayPts',
    'Home_Overall_Last5_Points', 'Away_Overall_Last5_Points',
    'Home_Overall_Last5_ShotsOTFor',
    'Away_Overall_Last5_YellowsFor'
]
print(df[kolumny_do_podgladu].tail(10))

           Date       HomeTeam          AwayTeam  Home_Last5_HomePts  \
9691 2026-03-03        Leeds_2      Sunderland_3                 1.4   
9692 2026-03-03  Bournemouth_1       Brentford_0                 1.6   
9693 2026-03-03      Everton_0         Burnley_4                 0.4   
9694 2026-03-03       Wolves_2       Liverpool_0                 1.0   
9695 2026-03-04    Newcastle_2      Man United_0                 1.2   
9696 2026-03-04     Man City_1   Nott'm Forest_0                 2.2   
9697 2026-03-04  Aston Villa_1         Chelsea_0                 1.4   
9698 2026-03-04       Fulham_3        West Ham_2                 2.0   
9699 2026-03-04     Brighton_0         Arsenal_0                 1.6   
9700 2026-03-05    Tottenham_0  Crystal Palace_1                 0.4   

      Away_Last5_AwayPts  Home_Overall_Last5_Points  \
9691                 0.4                        1.0   
9692                 2.4                        1.8   
9693                 1.0                  

In [99]:
df = df.sort_values(by='Date').reset_index(drop=True)
df['H2H_Home_Pts']=df.groupby(['HomeTeam', 'AwayTeam'])['HomePoints'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())
df['H2H_Home_GF']=df.groupby(['HomeTeam', 'AwayTeam'])['FTHG'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())
df['H2H_Home_GA']=df.groupby(['HomeTeam', 'AwayTeam'])['FTAG'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())



In [100]:
h2h_home = df[['Date','HomeTeam', 'AwayTeam', 'HomePoints','FTHG','FTAG']].copy()
h2h_home.columns = ['Date', 'Team', 'Opponent', 'Points', 'GF', 'GA']

h2h_away = df[['Date', 'AwayTeam', 'HomeTeam', 'AwayPoints','FTAG','FTHG']].copy()
h2h_away.columns = ['Date', 'Team', 'Opponent', 'Points','GF', 'GA']

h2h_all = pd.concat([h2h_home, h2h_away]).sort_values(by=['Team', 'Opponent','Date']).reset_index(drop=True)
h2h_all['H2H_Overall_Pts']= h2h_all.groupby(['Team','Opponent'])['Points'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
h2h_all['H2H_Overall_GF']=h2h_all.groupby(['Team','Opponent'])['GF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
h2h_all['H2H_Overall_GA']=h2h_all.groupby(['Team','Opponent'])['GA'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

h2h_all = h2h_all[['Date','Team','Opponent','H2H_Overall_Pts','H2H_Overall_GF','H2H_Overall_GA']]



In [101]:
df = df.merge(h2h_all, left_on=['Date', 'HomeTeam','AwayTeam'], right_on=['Date','Team','Opponent'], how='left')
df.rename(columns={
    'H2H_Overall_Pts': 'Home_H2H_Overall_Pts',
    'H2H_Overall_GF': 'Home_H2H_Overall_GF',
    'H2H_Overall_GA': 'Home_H2H_Overall_GA',
}, inplace=True)

In [102]:
print(df[df['HomeTeam'] == 'Liverpool'][['Date', 'HomeTeam', 'AwayTeam', 'H2H_Home_Pts', 'Home_H2H_Overall_Pts']].tail(10))

Empty DataFrame
Columns: [Date, HomeTeam, AwayTeam, H2H_Home_Pts, Home_H2H_Overall_Pts]
Index: []


In [103]:
print(df[['Date','HomeTeam','AwayTeam','Home_DaysOfRest','Away_DaysOfRest']].tail(10))

           Date       HomeTeam          AwayTeam  Home_DaysOfRest  \
9691 2026-03-03  Bournemouth_1       Brentford_0              3.0   
9692 2026-03-03      Everton_0         Burnley_4              3.0   
9693 2026-03-03        Leeds_2      Sunderland_3              3.0   
9694 2026-03-03       Wolves_2       Liverpool_0              4.0   
9695 2026-03-04       Fulham_3        West Ham_2              3.0   
9696 2026-03-04  Aston Villa_1         Chelsea_0              5.0   
9697 2026-03-04     Brighton_0         Arsenal_0              3.0   
9698 2026-03-04    Newcastle_2      Man United_0              4.0   
9699 2026-03-04     Man City_1   Nott'm Forest_0              4.0   
9700 2026-03-05    Tottenham_0  Crystal Palace_1              4.0   

      Away_DaysOfRest  
9691              3.0  
9692              3.0  
9693              3.0  
9694              3.0  
9695              4.0  
9696              3.0  
9697              3.0  
9698              3.0  
9699              3.0  


*TODO: SYSTEM ELO*

In [104]:
elo_dict={}
K_FACTOR = 20
HFA =70
SCALE_FACTOR = 400

def expected_result(elo_a, elo_b):
    return 1 / (1+10**((elo_b-elo_a)/ SCALE_FACTOR))

def get_g_modifier(goal_diff):
    if goal_diff <=1:
        return 1.0
    elif goal_diff == 2:
        return 1.5
    else:
        return (11 + goal_diff) / 8.0

In [105]:
home_elos_before = []
away_elos_before = []

for index, row in df.iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']

    if home not in elo_dict:
        elo_dict[home] = 1500 if row['Season']=='2000/2001' else 1350
    if away not in elo_dict:
        elo_dict[away] = 1500 if row['Season']=='2000/2001' else 1350

    elo_h = elo_dict[home]
    elo_a = elo_dict[away]
    home_elos_before.append(elo_h)
    away_elos_before.append(elo_a)

    elo_h_adj = elo_h + HFA

    exp_h= expected_result(elo_h_adj, elo_a)
    exp_a= expected_result(elo_a, elo_h_adj)

    goals_h = row['FTHG']
    goals_a = row['FTAG']
    goals_diff = abs(goals_h - goals_a)

    G = get_g_modifier(goals_diff)

    if goals_h > goals_a:
        res_h, res_a = 1.0, 0.0
    elif goals_h == goals_a:
        res_h, res_a = 0.5, 0.5
    else:
        res_h, res_a = 0.0, 1.0

    elo_dict[home] = elo_h + K_FACTOR * G * (res_h - exp_h)
    elo_dict[away] = elo_a + K_FACTOR * G * (res_a - exp_a)

df['Home_ELO'] = home_elos_before
df['Away_ELO'] = away_elos_before

In [106]:
print(df[['Date','HomeTeam','AwayTeam','FTHG','FTAG','Home_ELO','Away_ELO']].tail(10))

           Date       HomeTeam          AwayTeam  FTHG  FTAG     Home_ELO  \
9691 2026-03-03  Bournemouth_1       Brentford_0   0.0   0.0  1543.482746   
9692 2026-03-03      Everton_0         Burnley_4   2.0   0.0  1523.193056   
9693 2026-03-03        Leeds_2      Sunderland_3   0.0   1.0  1446.569499   
9694 2026-03-03       Wolves_2       Liverpool_0   2.0   1.0  1368.313997   
9695 2026-03-04       Fulham_3        West Ham_2   0.0   1.0  1520.782142   
9696 2026-03-04  Aston Villa_1         Chelsea_0   1.0   4.0  1612.636585   
9697 2026-03-04     Brighton_0         Arsenal_0   0.0   1.0  1554.621039   
9698 2026-03-04    Newcastle_2      Man United_0   2.0   1.0  1539.318509   
9699 2026-03-04     Man City_1   Nott'm Forest_0   2.0   2.0  1743.416212   
9700 2026-03-05    Tottenham_0  Crystal Palace_1   1.0   3.0  1446.468017   

         Away_ELO  
9691  1549.955242  
9692  1333.614742  
9693  1430.753787  
9694  1680.895408  
9695  1425.829772  
9696  1613.957990  
9697  1759.3

In [107]:
df['HomeTeam'] = df['HomeTeam'].str.split('_').str[0]
df['AwayTeam'] = df['AwayTeam'].str.split('_').str[0]

**FINAL DATA CLEARING**

In [110]:
cols_form = [col for col in df.columns if 'Last5' in col or 'H2H' in col]

for col in cols_form:
    low_baseline = df[col].quantile(0.25)
    df[col] = df[col].fillna(low_baseline)

cols_h2h_pts = [col for col in df.columns if 'H2H' in col and 'Pts' in col]
cols_h2h_goals =  [col for col in df.columns if 'H2H' in col and ('GF' in col or 'GA' in col)]

for col in cols_h2h_pts:
    df[col] = df[col].fillna(1.0)

for col in cols_h2h_goals:
    df[col] = df[col].fillna(1.0)



In [112]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Series([], dtype: int64)
